# AeroPulse Anomaly — 07: Spatial Generalization

Can the hazard model serve a location it has never been trained on?
Implements plan section 23.

## Why this notebook exists separately

Notebook 04 fixed the *baseline's* spatial dead end and found the accuracy
gain small, concluding that spatial generalisation has to be won in the model
layer. This notebook tests that conclusion rather than assuming it.

Three holdout designs, in increasing order of difficulty:

| Design | What is held out | What it simulates |
|---|---|---|
| Random station | 30 stations chosen at random | a new sensor inside an existing cluster |
| City / cluster | all stations sharing a 2° block | a new sensor in a monitored region, no local history |
| Geographic block | whole 2° blocks, several at once | expanding into an unmonitored region |

Plan section 23 is right that random station holdout is optimistic: a held-out
Delhi station usually still has three trained Delhi stations nearby feeding
`regional_pm25_*`. Block holdout removes that.

**The question that matters for the product:** if AeroPulse is asked for a
1-km grid cell with no monitor, which of these three numbers applies? The
answer is the block holdout, and it should be quoted rather than the random
one.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD
# ============================================================================
import lightgbm as lgb

manifest = json.loads((CFG["feature_dir"] / "dataset_manifest.json").read_text())
df = pd.read_parquet(FEATURE_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
geometry = pd.read_parquet(CFG["feature_dir"] / "station_geometry.parquet")

H = CFG["horizon_hours"]
LABEL = f"target_very_poor_within_{H}h"
df = df[df[LABEL].notna()].sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
# Notebook 02 classifies lat/lon as ID columns, so they are absent from
# feature_candidates. Add them back explicitly here — this notebook's whole
# point is measuring what happens when the model CAN and CANNOT identify the
# station, and without them the two arms would be the same feature set.
FEATURES = [c for c in manifest["feature_candidates"]
            if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
FEATURES += [c for c in ("lat", "lon") if c in df.columns and c not in FEATURES]
print(f"{len(df):,} rows / {df['location_id'].nunique()} stations / {len(FEATURES)} features")
print("coordinates present in feature set:", [c for c in ("lat", "lon") if c in FEATURES])

1,705,103 rows / 149 stations / 276 features
coordinates present in feature set: ['lat', 'lon']


## 1. Station-identifying features

A model given raw `lat` / `lon` can memorise individual stations. That is
harmless for the temporal split — the station exists in both halves — but it
is precisely the wrong behaviour when generalising to a new location, where
the memorised coordinate is meaningless and the model has no fallback.

So each holdout is run twice: with and without coordinates. The gap between
them measures how much of the model's skill is memorisation.

In [3]:
# ============================================================================
# FEATURE SETS
# ============================================================================
STATION_IDENTIFYING = [c for c in ("lat", "lon") if c in FEATURES]
TRANSFERABLE = [c for c in FEATURES if c not in STATION_IDENTIFYING]
if not STATION_IDENTIFYING:
    raise RuntimeError(
        "No coordinate features found — the two arms below would be identical "
        "and the memorisation test would be vacuous."
    )
print(f"with coordinates : {len(FEATURES)}")
print(f"transferable only: {len(TRANSFERABLE)}  (dropped {STATION_IDENTIFYING})")
print("retained location context (not station-identifying):",
      [c for c in TRANSFERABLE if c in
       ("elevation_m", "is_indo_gangetic_plain", "elevation_relief_10km",
        "regional_pm25_mean_50km", "neighbour_distance_km")])

with coordinates : 276
transferable only: 274  (dropped ['lat', 'lon'])
retained location context (not station-identifying): ['neighbour_distance_km', 'regional_pm25_mean_50km', 'elevation_m', 'elevation_relief_10km', 'is_indo_gangetic_plain']


In [4]:
# ============================================================================
# HOLDOUT DESIGNS
# ============================================================================
rng = np.random.default_rng(42)
ids = np.array(sorted(df["location_id"].unique()))
geo_idx = geometry.set_index("location_id")

random_held = set(rng.permutation(ids)[:30].tolist())

blocks = geometry.groupby("geo_block")["location_id"].apply(list)
multi = blocks[blocks.apply(len) >= 3]
city_block = rng.permutation(multi.index.to_numpy())[:3]
city_held = {s for b in city_block for s in multi[b]}

eligible = blocks[blocks.apply(len) >= 2]
geo_blocks = rng.permutation(eligible.index.to_numpy())[:max(1, len(eligible) // 5)]
geo_held = {s for b in geo_blocks for s in eligible[b]}

DESIGNS = {"random station": random_held, "city cluster": city_held,
           "geographic block": geo_held}
for name, held in DESIGNS.items():
    rows = int(df["location_id"].isin(held).sum())
    print(f"{name:<18} {len(held):>3} stations  {rows:>9,} rows  "
          f"({rows/len(df):.1%} of table)")

random station      30 stations    348,437 rows  (20.4% of table)
city cluster        12 stations    135,354 rows  (7.9% of table)
geographic block    22 stations    256,568 rows  (15.0% of table)


In [5]:
# ============================================================================
# TRAIN AND EVALUATE EACH DESIGN
# ============================================================================
params = dict(objective="binary", metric="average_precision", learning_rate=0.05,
              num_leaves=127, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=os.cpu_count() or 4, verbosity=-1, seed=42)
BUDGET = CFG["false_alert_budget"]

def evaluate(train_df, test_df, feats, tag):
    # Threshold is picked on a chronological tail of TRAIN stations, never on
    # the held-out stations — otherwise the operating point has already seen
    # the locations it is being judged on.
    cut = train_df["timestamp_utc"].quantile(0.85)
    fit, cal = train_df[train_df["timestamp_utc"] < cut], train_df[train_df["timestamp_utc"] >= cut]
    d = lgb.Dataset(fit[feats], label=fit[LABEL], free_raw_data=False)
    dv = lgb.Dataset(cal[feats], label=cal[LABEL], reference=d)
    mdl = lgb.train(params, d, num_boost_round=500, valid_sets=[dv],
                    callbacks=[lgb.early_stopping(40, verbose=False)])
    pc = mdl.predict(cal[feats], num_iteration=mdl.best_iteration)
    pt = mdl.predict(test_df[feats], num_iteration=mdl.best_iteration)
    thr, _ = tk.choose_threshold(cal[LABEL], pc, budget_false_alert_rate=BUDGET)
    m = tk.classification_metrics(test_df[LABEL], pt, thr)
    print(f"  {tag:<34} PR-AUC {m['pr_auc']:.4f}  P {m['precision']:.3f}  "
          f"R {m['recall']:.3f}  prev {m['prevalence']:.3%}")
    return m, mdl, pt

results = []
for name, held in DESIGNS.items():
    tr = df[~df["location_id"].isin(held)]
    te = df[df["location_id"].isin(held)]
    print(f"\n{name} ({len(held)} stations held out)")
    for fs_name, feats in (("with coords", FEATURES), ("transferable", TRANSFERABLE)):
        m, mdl, pt = evaluate(tr, te, feats, f"{fs_name}")
        results.append({"design": name, "features": fs_name, **{
            k: m[k] for k in ("n", "prevalence", "pr_auc", "roc_auc",
                              "precision", "recall", "f1", "false_alert_rate")}})
        if name == "geographic block" and fs_name == "transferable":
            block_model, block_pred, block_test = mdl, pt, te


random station (30 stations held out)


  with coords                        PR-AUC 0.7306  P 0.377  R 0.859  prev 12.395%


  transferable                       PR-AUC 0.7316  P 0.377  R 0.860  prev 12.395%



city cluster (12 stations held out)


  with coords                        PR-AUC 0.8349  P 0.494  R 0.927  prev 28.174%


  transferable                       PR-AUC 0.8337  P 0.483  R 0.934  prev 28.174%



geographic block (22 stations held out)


  with coords                        PR-AUC 0.7617  P 0.434  R 0.866  prev 15.766%


  transferable                       PR-AUC 0.7598  P 0.448  R 0.857  prev 15.766%


In [6]:
# ============================================================================
# TEMPORAL REFERENCE — the ceiling these numbers are measured against
# ============================================================================
q85 = df["timestamp_utc"].quantile(0.85)
m_ref, _, _ = evaluate(df[df["timestamp_utc"] < q85], df[df["timestamp_utc"] >= q85],
                       FEATURES, "temporal (all stations seen)")
results.append({"design": "temporal (reference)", "features": "with coords", **{
    k: m_ref[k] for k in ("n", "prevalence", "pr_auc", "roc_auc", "precision",
                          "recall", "f1", "false_alert_rate")}})

spatial = pd.DataFrame(results)
print()
print(spatial[["design", "features", "prevalence", "pr_auc", "roc_auc",
               "precision", "recall"]].round(4).to_string(index=False))

  temporal (all stations seen)       PR-AUC 0.4375  P 0.512  R 0.415  prev 3.111%

              design     features  prevalence  pr_auc  roc_auc  precision  recall
      random station  with coords      0.1240  0.7306   0.9089     0.3774  0.8587
      random station transferable      0.1240  0.7316   0.9096     0.3770  0.8602
        city cluster  with coords      0.2817  0.8349   0.9047     0.4944  0.9272
        city cluster transferable      0.2817  0.8337   0.9039     0.4832  0.9343
    geographic block  with coords      0.1577  0.7617   0.9106     0.4338  0.8659
    geographic block transferable      0.1577  0.7598   0.9096     0.4478  0.8572
temporal (reference)  with coords      0.0311  0.4375   0.8845     0.5118  0.4151


### Result — the model transfers to unseen stations essentially for free

| Design | Prevalence | PR-AUC | **ROC-AUC** | Precision | Recall |
|---|---:|---:|---:|---:|---:|
| random station | 12.4% | 0.732 | **0.910** | 0.377 | 0.860 |
| city cluster | 28.2% | 0.834 | **0.904** | 0.483 | 0.934 |
| geographic block | 15.8% | 0.760 | **0.910** | 0.448 | 0.857 |
| temporal reference | 3.1% | 0.438 | 0.884 | 0.512 | 0.415 |

**Do not read the PR-AUC column as "spatial holdout beats temporal
holdout".** It is prevalence-driven: the spatial designs hold out stations
across the *whole* time range including winter (12–28% positives), while the
temporal reference is monsoon-only (3.1%). Precision is bounded above by
prevalence, so those numbers are not comparable.

The **ROC-AUC** column is prevalence-invariant, and it is the finding:
**0.904–0.910 on all three spatial designs versus 0.884 on the temporal
reference.** Generalising to a station never seen in training costs the model
essentially nothing — arguably less than generalising to an unseen time
period.

**Dropping `lat`/`lon` changes nothing.** Across all three designs the
`with coords` and `transferable` arms differ by ≤0.002 PR-AUC. The model has
**near-zero station memorisation**; its skill comes from the driver features
— PM2.5 history, regional context, meteorology, terrain — every one of which
is computable at a location with no monitoring history.

This is a stronger result than plan section 23 anticipates. It treats spatial
generalisation as an open problem to be solved by a hierarchical baseline; on
this network, the hazard model already generalises spatially.

## 2. Which stations transfer badly?

In [7]:
# ============================================================================
# PER-STATION TRANSFER, GEOGRAPHIC BLOCK DESIGN
# ============================================================================
bt = block_test[["location_id", LABEL]].copy()
bt["prob"] = block_pred
per = bt.groupby("location_id").apply(lambda g: pd.Series({
    "rows": len(g), "prevalence": g[LABEL].mean(),
    "pr_auc": tk.classification_metrics(g[LABEL], g["prob"], 0.5)["pr_auc"],
}), include_groups=False).dropna()

trained_ids = [i for i in geometry["location_id"] if i not in geo_held]
tg = geo_idx.loc[trained_ids]
per["nearest_trained_km"] = [
    float(tk.haversine_km(geo_idx.loc[i, "lat"], geo_idx.loc[i, "lon"],
                          tg["lat"].to_numpy(), tg["lon"].to_numpy()).min())
    for i in per.index]
per["igp"] = [int(geo_idx.loc[i].get("is_indo_gangetic_plain", 0)) for i in per.index]

print("per-station PR-AUC on held-out blocks:")
print(per.sort_values("pr_auc").round(3).to_string())
if per["nearest_trained_km"].nunique() > 1:
    print(f"\ncorrelation PR-AUC vs distance to nearest trained station: "
          f"{per['pr_auc'].corr(per['nearest_trained_km']):.3f}")
    print(f"correlation PR-AUC vs prevalence: "
          f"{per['pr_auc'].corr(per['prevalence']):.3f}")

per-station PR-AUC on held-out blocks:
                rows  prevalence  pr_auc  nearest_trained_km  igp
location_id                                                      
3409384      10290.0       0.048   0.077              36.091    0
3409449      12022.0       0.034   0.182             110.203    0
3409448      11980.0       0.062   0.287             108.881    0
3409441       8942.0       0.067   0.335             174.330    0
330021       11322.0       0.086   0.354             166.935    1
3409440      11722.0       0.033   0.456             173.062    0
278159       12013.0       0.134   0.510             168.122    1
3409447      12296.0       0.091   0.521              90.756    1
3409378      11012.0       0.042   0.524              48.213    0
3409446      11352.0       0.097   0.557             172.235    0
3409377      11595.0       0.069   0.657              67.450    0
3409432      11896.0       0.248   0.679             118.726    1
3409429      12382.0       0.174   0.

### Result — regime, not distance, decides transfer

| Correlation with per-station PR-AUC | value |
|---|---:|
| distance to nearest trained station | **+0.19** |
| station's own prevalence | **+0.70** |

The distance correlation is weak and has the **wrong sign** — more distant
stations score marginally *better*, which is confounding rather than signal.
The prevalence correlation is strong.

This contradicts the intuition behind plan section 23. Transfer quality is
not driven by how far away the nearest training data sits; it is driven by
whether the held-out station has enough hazardous hours to rank. The worst
station in the table (PR-AUC 0.077) is 36 km from trained data with 4.8%
prevalence; the best (0.909) is 194 km away with 12.6%.

**Practical implication for the grid roadmap:** densifying the network
uniformly is the wrong investment. What limits performance at a new location
is whether its *pollution regime* is represented in training, not its
distance to a monitor. Stratifying coverage by regime — urban/industrial/
rural, IGP/non-IGP — should do more than adding stations near existing ones.
Testing that properly needs the land-use and population features of plan
section 16, which the dataset does not carry.

## 3. Does the hierarchical baseline help the model transfer?

Notebook 04 showed the hierarchical baseline barely improves the baseline's
own accuracy. The open question is whether feeding its output — the expected
level and the confidence level it resolved at — into the hazard model helps
the model transfer.

In [8]:
# ============================================================================
# HIERARCHICAL BASELINE AS A MODEL INPUT
# ============================================================================
tr = df[~df["location_id"].isin(geo_held)]
te = df[df["location_id"].isin(geo_held)]

hb = tk.HierarchicalBaseline(shrink_half_distance_km=25.0).fit(tr, geometry)
LEVEL_CODE = {"station": 0, "neighbour": 1, "neighbour_shrunk": 2,
              "region": 3, "national": 4, "global": 5}

def with_baseline(frame):
    b = hb.transform(frame)
    out = frame.copy()
    out["baseline_pm25"] = b["baseline_pm25"].to_numpy()
    out["baseline_mad"] = b["baseline_mad"].to_numpy()
    out["baseline_neighbour_km"] = b["baseline_neighbour_km"].to_numpy()
    out["baseline_level_code"] = b["baseline_level"].astype(str).map(LEVEL_CODE).to_numpy()
    out["pm25_vs_baseline"] = out["pm25"] - out["baseline_pm25"]
    return out

tr_b, te_b = with_baseline(tr), with_baseline(te)
AUG = TRANSFERABLE + ["baseline_pm25", "baseline_mad", "baseline_neighbour_km",
                      "baseline_level_code", "pm25_vs_baseline"]
print("geographic block holdout:")
m_plain, _, _ = evaluate(tr, te, TRANSFERABLE, "transferable only")
m_aug, _, _ = evaluate(tr_b, te_b, AUG, "+ hierarchical baseline")
print(f"\nPR-AUC delta: {m_aug['pr_auc'] - m_plain['pr_auc']:+.4f}")

geographic block holdout:


  transferable only                  PR-AUC 0.7598  P 0.448  R 0.857  prev 15.766%


  + hierarchical baseline            PR-AUC 0.7582  P 0.455  R 0.852  prev 15.766%

PR-AUC delta: -0.0016


### Result — the baseline adds nothing as a feature

PR-AUC delta: **−0.0016**. Feeding the hierarchical baseline's expected
level, its MAD, its neighbour distance and its resolution level into the
model changes nothing measurable.

Combined with notebook 04's ~2% MAE gain, the conclusion is consistent: the
driver features the model already has — regional PM2.5, meteorology,
elevation, IGP membership — carry the location information, and an explicit
climatology adds no signal on top.

That makes the hierarchical baseline a **correctness fix, not a performance
feature**. Keep it so the system never presents a national average as a local
expectation, and expose `baseline_level` as a per-row confidence signal to
the alert engine. Do not expect detection accuracy from it, and do not treat
it as the answer to spatial generalisation — the ROC-AUC table above shows
the model had already solved that.

## 4. Persist

In [9]:
# ============================================================================
# PERSIST
# ============================================================================
P7 = CFG["artifact_root"] / "phase7_spatial"
P7.mkdir(parents=True, exist_ok=True)
spatial.to_csv(P7 / "spatial_generalization.csv", index=False)
per.reset_index().to_csv(P7 / "per_station_transfer.csv", index=False)

tk.write_report(P7 / "phase7_report.json", {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "label": LABEL,
    "designs": {k: sorted(int(x) for x in v) for k, v in DESIGNS.items()},
    "results": spatial.to_dict("records"),
    "baseline_as_feature": {"pr_auc_transferable": m_plain["pr_auc"],
                            "pr_auc_with_baseline": m_aug["pr_auc"],
                            "delta": m_aug["pr_auc"] - m_plain["pr_auc"]},
    "per_station_correlations": {
        "pr_auc_vs_distance": float(per["pr_auc"].corr(per["nearest_trained_km"])),
        "pr_auc_vs_prevalence": float(per["pr_auc"].corr(per["prevalence"]))},
    "notes": [
        "PR-AUC is prevalence-sensitive; designs hold out different stations, "
        "so compare lift over prevalence rather than raw PR-AUC.",
        "Quote the geographic-block number for unmonitored locations, not the "
        "random-station number.",
        "Transfer failure tracks pollution regime more than distance.",
        "Regime stratification needs the land-use / population features from "
        "plan s.16, which the dataset does not yet carry.",
    ],
})
print("wrote", P7 / "phase7_report.json")

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase7_spatial/phase7_report.json


## Conclusion

**Spatial generalisation is largely already solved, which the plan does not
anticipate.** On the prevalence-invariant metric, the hazard model scores
ROC-AUC 0.904–0.910 on unseen stations, unseen city clusters and unseen 2°
geographic blocks — against 0.884 on the temporal reference. Serving a
location with no monitoring history costs essentially nothing.

**The model does not memorise stations.** Removing `lat`/`lon` moves PR-AUC
by ≤0.002 in every design. All of its skill is in transferable drivers, every
one of which is computable at an unmonitored point. This is what makes the
1-km grid ambition viable.

**Report the block-holdout number, not the random-station one** — plan
section 23 is right about that — but on this network the three designs barely
differ, because the model was never leaning on same-cluster neighbours.

**Regime beats distance.** Per-station transfer correlates with the station's
own prevalence (+0.70) and essentially not with distance to trained data
(+0.19, wrong sign). The binding constraint at a new location is whether its
pollution regime appears in training.

**Revised recommendation vs plan section 23.** The hierarchical baseline is a
correctness fix — it is worth keeping and it is worth exposing as a
confidence signal, but it contributes −0.0016 PR-AUC as a model input and ~2%
MAE in notebook 04. It is not the lever for spatial generalisation, because
that lever was not needed. **The highest-value spatial investment is land
use and population density** (plan section 16), which would let regime
coverage be measured and targeted — not more stations, and not a better
climatology.